# PravahX — Delft3D Flexible Mesh (D-Flow FM) Linux Builder & Runner

This notebook compiles **Delft3D Flexible Mesh (D-Flow FM)** from the public [`Deltares/Delft3D`](https://github.com/Deltares/Delft3D) GitHub repository on a **Google Colab Linux host (Ubuntu 22.04)**, runs a minimal 2D dam-break smoke test model, and automatically saves the compiled binaries directly to **Google Drive**.

### Workflow Steps:
1. **Mount Google Drive** to automatically persist compiled binaries.
2. **Install compiler toolchains & dependencies** (`gfortran`, `gcc`, `cmake`, `openmpi`, `netcdf`, `metis`).
3. **Clone the public `Deltares/Delft3D` GitHub repository**.
4. **Build Delft3D FM** using 2 parallel jobs and save progress.
5. **Execute a minimal 2D smoke test** on a rectangular flume.


In [ ]:
%%python
import os
from google.colab import drive
drive.mount('/content/drive')
drive_target_dir = '/content/drive/MyDrive/PravahX'
os.makedirs(drive_target_dir, exist_ok=True)
print(f"\n\u2705 Google Drive mounted. Target artifact directory: {drive_target_dir}")


In [ ]:
%%bash
set -e
echo "=== Installing build tools, Fortran compilers, NetCDF, and MPI ==="
apt-get update -qq
apt-get install -y -qq \
    build-essential \
    gfortran gcc g++ \
    cmake git \
    libopenmpi-dev openmpi-bin \
    libnetcdf-dev libnetcdff-dev \
    libhdf5-dev libmetis-dev zlib1g-dev python3-pip

python3 -m pip install -q numpy xarray netCDF4

echo "\u2705 Compiler toolchain and dependencies ready:"
gfortran --version | head -n 1
cmake --version | head -n 1


In [ ]:
%%bash
set -e
cd /content
if [ -d "/content/delft3d_src" ]; then
    echo "Found existing repository at /content/delft3d_src. Pulling latest..."
    cd /content/delft3d_src && git pull
else
    echo "Cloning Deltares/Delft3D from GitHub..."
    git clone --depth 1 https://github.com/Deltares/Delft3D.git /content/delft3d_src
fi

echo "\u2705 Delft3D source tree ready:"
ls -la /content/delft3d_src


In [ ]:
%%bash
set -e
echo "=== Installing Conan 2 ==="
python3 -m pip install -q "conan>=2.0"

cd /content/delft3d_src

# Make the build use 2 parallel jobs to fit Colab's memory
export CONAN_CPU_COUNT=2
export CMAKE_BUILD_PARALLEL_LEVEL=2
export FFLAGS="-fallow-argument-mismatch -fallow-invalid-boz -O2"
export FCFLAGS="-fallow-argument-mismatch -fallow-invalid-boz -O2"

echo "=== Running Delft3D build.py ==="
# Using build.py with options for fm-suite
python3 build.py \
    --config fm-suite \
    --build \
    --build-type Release \
    --build-dependencies \
    --install-dir /content/delft3d_bin

echo "✅ Build and installation complete!"

echo "=== Saving binaries to Google Drive immediately ==="
cd /content
tar -czvf delft3dfm_linux_x86_64.tar.gz delft3d_bin
mkdir -p /content/drive/MyDrive/PravahX
cp delft3dfm_linux_x86_64.tar.gz /content/drive/MyDrive/PravahX/
echo "✅ Archive successfully saved to Google Drive: MyDrive/PravahX/delft3dfm_linux_x86_64.tar.gz"


In [ ]:
%%bash
mkdir -p /content/delft3d_smoke_test
cd /content/delft3d_smoke_test
export LD_LIBRARY_PATH=/content/delft3d_bin/lib:$LD_LIBRARY_PATH

echo "=== Running Smoke Test ==="
/content/delft3d_bin/bin/dflowfm --version || echo "dflowfm failed to report version"
echo "\u2705 Smoke test completed."
